# RAG Implementation  using Hybrid Search 

In [ ]:
#  Install dependencies
!uv add  langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers rank-bm25 huggingface_hub

In [ ]:
# Import the libraries needed for loading data, building retrievers, and creating the RAG chain
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


In [ ]:
# Load environment variables and configure the models used in the workflow
load_dotenv(override=True)

HF_API_KEY = os.getenv("HF_API_KEY")  # Hugging Face token
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
BASE_URL = os.getenv("MODEL_BASE_URL")

# Embedding model for semantic retrieval
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Reranker model for improving retrieval quality
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"


Data Source used    
http://mlg.ucd.ie/datasets/bbc.html?source=post_page-----4340b55fef22---------------------------------------


In [ ]:

## -------- 1. Document Processing (loading documents)
folder_path = "C:\\Users\\pc\\Downloads\\bbc-fulltext\\bbc\\tech"

# Load all .txt files in that folder
loader = DirectoryLoader(
    folder_path,
    glob="**/*.txt",          # matches all txt files (even in subfolders)
    loader_cls=TextLoader
)

# Load documents
documents = loader.load()

print(f"Loaded {len(documents)} files")

In [ ]:
## -------- 2. Chunking (splitting documents into chunks)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)
print(len(chunks))

In [ ]:
## -------- 3. Embeddings
embeddings = HuggingFaceEndpointEmbeddings(
    model=EMBEDDING_MODEL, huggingfacehub_api_token=HF_API_KEY
)

# vector retriever for  search 
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})


# BM25 retriever for keyword search
bm25_retriever = BM25Retriever.from_documents(documents=chunks)
bm25_retriever.k = 10


In [ ]:

## ------- 4. Hybrid  (BM25 + Vector Search )
# weight=0.5 means 50% vector, 50% keyword. Adjust based on  data.
ensemble_retriever = EnsembleRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever,
    ],
    weights=[0.6, 0.4],
)


In [ ]:
## ------- 5. Reranking

# cross encoder -
"""A cross encoder in RAG hybrid search is a **model that jointly processes the query and a candidate document to output a precise relevance score for re-ranking retrieved results.** """
cross_encoder = HuggingFaceCrossEncoder(
    model_name="cross-encoder/ms-marco-MiniLM-L6-v2",
    model_kwargs={
        "device": "cpu", 
    },
)


# reranker
reranker = CrossEncoderReranker(model=cross_encoder, top_n=5)

In [ ]:
## ------- 6. Context Compression
compression_retriever = ContextualCompressionRetriever(
    base_retriever=ensemble_retriever,  # hybrid retriever 
    base_compressor=reranker  
)

docs = compression_retriever.invoke("How many stores opened in china this year?")
docs

In [ ]:
## ------- 7. Prompt engineerging 
prompt = ChatPromptTemplate.from_template(
"""
You are an AI Assistant that follows instructions extremely well.
Please be truthful and give direct answers. Please tell 'I don't know' if user query is not in CONTEXT

Context:
{context}

Question:
{question}

Answer:
"""
)

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini",base_url = BASE_URL,api_key=GITHUB_TOKEN)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": compression_retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# invoking chain
query = "How many cafes were closed in 2004 in China?"
response = rag_chain.invoke(query)
response

In [ ]:
## had to implement return soure in reponse ? 